# DisasterLens on a Colab GPU

This notebook pulls the repository into the Colab runtime, installs it, and verifies the GPU.

In [9]:
import os
import subprocess
import base64
from getpass import getpass
from pathlib import Path

github_token = os.environ.get("GITHUB_TOKEN")
if not github_token:
    try:
        from google.colab import userdata
        github_token = userdata.get("GITHUB_TOKEN")
    except Exception:
        github_token = None
if not github_token:
    github_token = getpass("GitHub token (input is hidden): ")
github_token = github_token.strip()
if not github_token:
    raise RuntimeError("No GitHub token was supplied.")

REPO_URL = os.environ.get("DISASTERLENS_REPO_URL", "https://github.com/kushc2004/disaster-lens.git")
REPO_DIR = Path("/content/disaster-lens")
git_env = os.environ.copy()
basic_auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
git_env.update({"GIT_CONFIG_COUNT": "1", "GIT_CONFIG_KEY_0": "http.extraHeader", "GIT_CONFIG_VALUE_0": f"Authorization: Basic {basic_auth}"})

if REPO_DIR.exists():
    if (REPO_DIR / ".git").exists():
        command = ["git", "-C", str(REPO_DIR), "pull"]
    else:
        import shutil
        shutil.rmtree(REPO_DIR)
        command = ["git", "clone", REPO_URL, str(REPO_DIR)]
else:
    command = ["git", "clone", REPO_URL, str(REPO_DIR)]
result = subprocess.run(command, env=git_env, capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(f"GitHub command failed ({result.returncode}):\n{result.stderr.strip()}")
print(result.stdout, end="")
%cd /content/disaster-lens

RuntimeError: GitHub command failed (128):
Cloning into '/content/disaster-lens'...
fatal: could not read Username for 'https://github.com': No such device or address

In [ ]:
%pip install -e .

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
assert torch.cuda.is_available(), "Select a Colab GPU kernel before running this notebook."

In [ ]:
# Current repository smoke check
!python scripts/create_smoke_bright.py
!python scripts/inspect_bright.py data=bright dataset.root=data/samples/bright_smoke
!pytest